<a href="https://colab.research.google.com/github/AlexVentura10/Projeto_Estacionamento/blob/main/C%C3%B3pia_de_Aula_8_BERT_classificador_de_trechos_de_livros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 8 - BERT classificador de trechos de livros

Nome: Alex Mateus da Silva Ventura

## Instruções:

Criar um classificar de trechos do dataset Machado de Assis como fazendo parte do livro correto (dentre 8 livros), utilizando Embeddings gerados por um BERT pré-treinado e uma MLP no token [CLS].

Objetivos:
- O modelo deve associar um trecho de livro ao livro correspondente (classificação).
- A modelagem do dataset, criação de chunks de texto, limpeza dos dados, split, é sua responsabilidade.

Detalhes:
- Deve-se implementar o próprio laço de treinamento. **Não usar frameworks de treinamento automático.**
- Utilizar o BertModel.from_pretrained e BertTokenizer.from_pretrained do Hugging Face para carregar o BERT pré-treinado.
- Não utilize outras classes da HuggingFace/Transformers alem da [BertModel](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertModel) e [BertTokenizer](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer).
- O BERT original do artigo pode ser difícil de rodar, tentar utilizar variações com menos parâmetros, ou pré-treinadas em PT-BR (como BERTimbau).
- Deixe para usar GPU somente quando tudo estiver validado.

Tarefas OPCIONAIS, para quem tem computação:
- Experimente com mudar o contexto e congelar ou não os parâmetros do BERT. Cuidado que o contexto é o maior fator para o peso computacional aqui.                                                                             
- A classificação usando livros_com_nomes deve ser mais fácil do que usando os livros_sem_nomes. Compare os resultados entre usar livros_com_nomes e livros_sem_nomes.           

Além do notebook com todas as saídas e com métricas de classificação, entregue uma breve discussão do que foi feito.
                                                                             

In [ ]:
import random
import torch
import torch.nn.functional as F
import numpy as np

In [ ]:
random.seed(123)
np.random.seed(123)
torch.manual_seed(123)

## Estudo BERT

### Tokenizer
-Testar com strings\
-Tokens especiais\
-Tokenize\
-Decode\
-Batch e padding


In [ ]:
from transformers import BertTokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# tokenização
tokenizer("Hello World").input_ids

[101, 7592, 2088, 102]

In [ ]:
# tokenização
tokenizer("Hello World")

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [ ]:
# tokens especiais

# tokens especiais
tokenizer.decode([0, 101, 102, 103])

'[PAD] [CLS] [SEP] [MASK]'

In [ ]:
tokenizer("Hello World Orthogonalization")

{'input_ids': [101, 7592, 2088, 28721, 3989, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [ ]:
# palavras são quebradas em subwords
tokenizer.tokenize("Hello World Orthogonalization")

['hello', 'world', 'orthogonal', '##ization']

In [ ]:
tokenizer.tokenize("Testando uma frase em português!")

['test',
 '##ando',
 'um',
 '##a',
 'fra',
 '##se',
 'em',
 'port',
 '##ug',
 '##ues',
 '!']

In [ ]:
input_str = "Machado de Assis é um escritor famoso."
tokenizer.tokenize(input_str)

['mach',
 '##ado',
 'de',
 'ass',
 '##is',
 'e',
 'um',
 'es',
 '##cr',
 '##itor',
 'fa',
 '##mos',
 '##o',
 '.']

In [ ]:
tokenizer(input_str)

{'input_ids': [101, 24532, 9365, 2139, 4632, 2483, 1041, 8529, 9686, 26775, 15660, 6904, 15530, 2080, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# tokenizador ajustado para PT-BR
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

In [ ]:
tokenizer(input_str)

{'input_ids': [101, 7445, 125, 11160, 253, 222, 4223, 4828, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.tokenize(input_str)

['Machado', 'de', 'Assis', 'é', 'um', 'escritor', 'famoso', '.']

In [ ]:
t = tokenizer(["Primeira sentença essa é muito maior",
               "Segunda sentença"], max_length=16, padding="max_length", truncation=False)
t.input_ids, t.attention_mask

([[101, 4085, 12385, 1921, 253, 785, 636, 102, 0, 0, 0, 0, 0, 0, 0, 0],
  [101, 3335, 12385, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
 [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [ ]:
for x in t.input_ids:
    print(tokenizer.decode(x))

[CLS] Primeira sentença essa é muito maior [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] Segunda sentença [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [ ]:
tokenizer.model_max_length

1000000000000000019884624838656

## Como attention mask funciona?

In [ ]:
Q = torch.tensor([[0, 1], [0, 1], [3, 3]]).float()
K = Q
V = Q

In [ ]:
Q, K, V

(tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]),
 tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]),
 tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]))

In [ ]:
K.transpose(1, 0)

tensor([[0., 0., 3.],
        [1., 1., 3.]])

In [ ]:
# Primeiro passo da atenção: relação NxN entre embeddings entre Q e K através de QK^T
scores = Q @ K.transpose(1, 0)
scores

tensor([[ 1.,  1.,  3.],
        [ 1.,  1.,  3.],
        [ 3.,  3., 18.]])

In [ ]:
scores[:, 2] = -10**6
scores

tensor([[ 1.0000e+00,  1.0000e+00, -1.0000e+06],
        [ 1.0000e+00,  1.0000e+00, -1.0000e+06],
        [ 3.0000e+00,  3.0000e+00, -1.0000e+06]])

In [ ]:
# Transformando em scores
attention = torch.nn.Softmax(dim=1)(scores)
attention

tensor([[0.5000, 0.5000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.5000, 0.5000, 0.0000]])

In [ ]:
# Scores ponderam V
attention @ V

tensor([[0., 1.],
        [0., 1.],
        [0., 1.]])

In [ ]:
# PERGUNTA: como eliminar a influência do embedding 3,3 (terceiro token)?

### Forward no modelo
Inspecionar modelo com torchinfo\
Visualizar saida do modelo

In [ ]:
!pip install torchinfo -q

In [ ]:
import torchinfo
from transformers import BertModel

In [ ]:
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
test_input = tokenizer("Hello BERT!", return_tensors='pt').input_ids
test_input, test_input.shape

(tensor([[  101, 15044, 22280,   241,  5054, 22321,   106,   102]]),
 torch.Size([1, 8]))

In [ ]:
output = model(input_ids=test_input)

In [ ]:
list(output.keys())

['last_hidden_state', 'pooler_output']

In [ ]:
# "estado final dos embeddings"
output.last_hidden_state.shape

output.last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

In [ ]:
# O que esta dentro da classe?
torchinfo.summary(model,
                  input_data=test_input,
                  depth=4)

Layer (type:depth-idx)                             Output Shape              Param #
BertModel                                          [1, 768]                  --
├─BertEmbeddings: 1-1                              [1, 8, 768]               --
│    └─Embedding: 2-1                              [1, 8, 768]               23,440,896
│    └─Embedding: 2-2                              [1, 8, 768]               1,536
│    └─Embedding: 2-3                              [1, 8, 768]               393,216
│    └─LayerNorm: 2-4                              [1, 8, 768]               1,536
│    └─Dropout: 2-5                                [1, 8, 768]               --
├─BertEncoder: 1-2                                 [1, 8, 768]               --
│    └─ModuleList: 2-6                             --                        --
│    │    └─BertLayer: 3-1                         [1, 8, 768]               --
│    │    │    └─BertAttention: 4-1                [1, 8, 768]               2,363,904
│    │   

In [ ]:
# O BERT TINY é a mesma coisa, só escalado para muito menos camadas!
model = BertModel.from_pretrained('prajjwal1/bert-tiny')

torchinfo.summary(model,
                  input_data=test_input,
                  depth=3)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer (type:depth-idx)                             Output Shape              Param #
BertModel                                          [1, 128]                  --
├─BertEmbeddings: 1-1                              [1, 8, 128]               --
│    └─Embedding: 2-1                              [1, 8, 128]               3,906,816
│    └─Embedding: 2-2                              [1, 8, 128]               256
│    └─Embedding: 2-3                              [1, 8, 128]               65,536
│    └─LayerNorm: 2-4                              [1, 8, 128]               256
│    └─Dropout: 2-5                                [1, 8, 128]               --
├─BertEncoder: 1-2                                 [1, 8, 128]               --
│    └─ModuleList: 2-6                             --                        --
│    │    └─BertLayer: 3-1                         [1, 8, 128]               198,272
│    │    └─BertLayer: 3-2                         [1, 8, 128]               198,272
├─BertPooler

## Dados NOVOS: machado_livros

Vamos usar UMA NOVA VERSÃO do dataset do Machado de Assis, dividido nos livros principais do autor.

Existem dois conjuntos: um com anonimização dos nomes e outro sem. A pasta livros_com_nomes deve ser uma fonte de dados mais fácil de classificar do que livros_sem_nomes.


In [ ]:
!wget -q https://github.com/unicamp-dl/ia025_machado_livros/releases/download/v1.0/livros_com_nomes.zip
!unzip -o -q livros_com_nomes.zip
!wget -q https://github.com/unicamp-dl/ia025_machado_livros/releases/download/v1.0/livros_sem_nomes.zip
!unzip -o -q livros_sem_nomes.zip

Exemplo de abertura de um trecho de um dos livros:

In [ ]:
import os

DATA_PATH = os.path.join("livros_sem_nomes", "helena.txt")

with open(DATA_PATH, "r") as data_file:
    lines = [line for line in data_file]

full_data = ' '.join(lines)
full_data[1000:2000]

'o lado materno\n descendia de uma das mais distintas famílias paulistas. Ele próprio exercera dois empregos,\n havendose com habilidade e decoro, do que lhe adveio a carta de conselho e a estima dos\n homens públicos. Sem embargo do ardor político do tempo, não estava ligado a nenhum dos\n dois partidos, conservando em ambos preciosas amizades, que ali se acharam na ocasião de o\n dar à sepultura. Tinha, entretanto, tais ou quais idéias políticas, colhidas nas fronteiras con\n servadoras e liberais, justamente no ponto em que os dois domínios podem confundirse. Se\n nenhuma saudade partidária lhe deitou a última pá de terra, matrona houve, e não só uma, que\n viu ir a enterrar com ele a melhor página da sua mocidade.\n A família do conselheiro compunhase de duas pessoas: um filho, o [NOME], e uma\n irmã, [NOME]. Contava esta cinqüenta e poucos anos; era solteira; vivera sempre com o\n irmão, cuja casa dirigia desde o falecimento da cunhada. [NOME] tinha vinte e sete anos, e era\n form

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

class MachadoLivrosDataset(Dataset):
    """
    PERGUNTAS DE DESENVOLVIMENTO:
    - Como foi feita a modelagem e o tratamento do dataset?

      Resposta: Lemos cada arquivo de texto bruto dos livros, removemos apenas
      espaços excessivos e aplicamos o tokenizador diretamente, fatiando o conteúdo
      em blocos sequenciais.

    - Por que foi preservado acentos, pontuações e letras maiúsculas?

      Resposta: Porque ao usar o BERTimbau 'cased', que é sensível à caixa e
      à pontuação da língua portuguesa, evitando o desalinhamento de domínio.
    """
    def __init__(self, data_dir, tokenizer, max_length=128, stride=64):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        livros_files = [f for f in os.listdir(data_dir) if f.endswith('.txt')]
        livros_files.sort()

        # Mapeia cada nome de livro para um ID numérico de classe (0 a 7)
        self.label2id = {filename.replace('.txt', ''): idx for idx, filename in enumerate(livros_files)}

        for filename in livros_files:
            livro_name = filename.replace('.txt', '')
            label_id = self.label2id[livro_name]

            file_path = os.path.join(data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # TRATAMENTO DOS DADOS: Normalização leve de espaços
            # (Mantém maiúsculas, pontuações e acentos nativos do português)
            text = " ".join(text.split())

            # TOKENIZAÇÃO: Converte o livro inteiro em IDs de tokens sem perdas por truncamento inicial
            full_encoding = tokenizer(
                text,
                add_special_tokens=False,
                return_attention_mask=False
            )['input_ids']

            # MODELAGEM DE CHUNKS (Janela Deslizante / Sliding Window):
            # Pergunta: Por que usar stride e janelas deslizantes?
            # Resposta: Para fatiar o livro em trechos menores que caibam no limite do BERT
            # sem descartar o conteúdo do meio das obras. O stride controla a sobreposição.
            target_len = max_length - 2  # Reserva espaço exato para os tokens [CLS] e [SEP]

            for i in range(0, len(full_encoding), stride):
                chunk_ids = full_encoding[i : i + target_len]
                if len(chunk_ids) < 30:  # Descarta pedaços residuais muito curtos no fim do livro
                    continue

                # INSERÇÃO DE TOKENS ESPECIAIS:
                # O [CLS] é adicionado no início (essencial para a tarefa de classificação via pooling)
                # O [SEP] marca o encerramento da sequência do trecho.
                input_ids = [tokenizer.cls_token_id] + chunk_ids + [tokenizer.sep_token_id]
                attention_mask = [1] * len(input_ids)

                # PADDING: Preenche com [PAD] até atingir o comprimento fixo (max_length=128)
                padding_length = max_length - len(input_ids)
                if padding_length > 0:
                    input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
                    attention_mask = attention_mask + [0] * padding_length

                self.samples.append({
                    'input_ids': torch.tensor(input_ids, dtype=torch.long),
                    'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
                    'label': torch.tensor(label_id, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# Instancia o dataset estruturado
DATA_DIR = "livros_com_nomes"
full_dataset = MachadoLivrosDataset(DATA_DIR, tokenizer, max_length=128, stride=64)

# DIVISÃO DOS DADOS (Split):
# Pergunta: Como os dados foram divididos?
# Resposta: 80% para treino, 10% para validação e 10% para teste, usando random_split com seed fixa.
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_data, val_data, test_data = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(123)
)

print(f"Total de Amostras Geradas: {len(full_dataset)}")
print(f"Treino: {len(train_data)} | Validação: {len(val_data)} | Teste: {len(test_data)}")

Total de Amostras Geradas: 32509
Treino: 26007 | Validação: 3250 | Teste: 3252


In [ ]:
""" TODO visualize se o dataloader faz sentido """
batch_size = 8  # Mantive um lote menor para economizar memória da GPU/CPU

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Testa se o dataloader está funcionando corretamente pegando um lote de amostra
sample_batch = next(iter(train_loader))
print("Formato dos IDs de entrada no lote:", sample_batch['input_ids'].shape)
print("Formato dos rótulos (labels) no lote:", sample_batch['label'].shape)

Formato dos IDs de entrada no lote: torch.Size([8, 128])
Formato dos rótulos (labels) no lote: torch.Size([8])


## Model

In [ ]:
import torch.nn as nn
from transformers import BertModel

class TextClassifier(torch.nn.Module):
    """
    PERGUNTAS DE DESENVOLVIMENTO:
    - Qual variação do BERT foi utilizada e por quê?

      Resposta: Utilizei o BERTimbau ('neuralmind/bert-base-portuguese-cased'),
      uma variação pré-treinada especificamente para o português, ideal para manter
      a acentuação e letras maiúsculas, em vez de usar o BERT original em inglês.

    - O BERT foi congelado ou treinado por completo (full fine-tuning)?

      Resposta: Não congelei o BERT; todos os parâmetros foram mantidos treináveis
      para permitir o ajuste fino completo da rede para a literatura de Machado de Assis.
    """
    def __init__(self, num_classes=8):
        super().__init__()
        # 1. Carrega o BERT pré-treinado em português (BERTimbau base)
        self.bert = BertModel.from_pretrained('neuralmind/bert-base-portuguese-cased')

        # 2. Cria a MLP de classificação usando o vetor do token [CLS] (dimensão oculta H = 768)
        # O BERT base possui hidden_size = 768
        hidden_size = self.bert.config.hidden_size

        # Classificador MLP simples com Dropout para evitar overfitting
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        # Passa os dados pelo BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # POOLING NO TOKEN [CLS]:
        # Pergunta: Por que usamos outputs.last_hidden_state[:, 0, :]?
        # Resposta: Porque o primeiro token da sequência (posição 0) é o token especial [CLS],
        # cuja representação agregada resume o contexto de todo o trecho de texto analisado.
        cls_representation = outputs.last_hidden_state[:, 0, :]

        # Passa pelo classificador MLP para prever a qual livro o trecho pertence
        logits = self.classifier(cls_representation)
        return logits

# Instancia o modelo
model = TextClassifier(num_classes=8)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Verifica se há uma GPU disponível e define o dispositivo para GPU se possível, caso contrário, usa a CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
import math

# Dispositivo em CPU ou GPU conforme disponibilidade do ambiente
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando o dispositivo: {device}")

model.to(device)

epochs = 3
lr = 2e-5  # Taxa de aprendizado padrão recomendada no artigo do BERT para fine-tuning
criterion = nn.CrossEntropyLoss() # Pergunta do professor: Por que usar CrossEntropyLoss? Resposta: É ideal para classificação multi-classe (os 8 livros).
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # Pergunta: Por que o AdamW? Resposta: É o otimizador padrão recomendado para Transformers por gerenciar melhor o weight decay.

for epoch in range(epochs):
    # --- FASE DE TREINAMENTO ---
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # PERGUNTA FREQUENTE DO PROFESSOR:
        # - Por que implementamos o laço de treinamento manualmente?
        # Resposta: Porque era um requisito explícito da atividade construir o próprio laço
        # do zero, sem utilizar frameworks de treinamento automático (como o Trainer).
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- FASE DE VALIDAÇÃO ---
    model.eval()
    total_val_loss = 0
    correct_preds = 0
    total_preds = 0

    # Pergunta: Para que serve o torch.no_grad()?
    # Resposta: Desativa o cálculo de gradientes durante a validação, economizando
    # memória VRAM e acelerando o processamento, já que não faremos ajuste de pesos aqui.
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            # Calcula acurácia comparando a classe com maior probabilidade (argmax) com o label real
            preds = torch.argmax(logits, dim=-1)
            correct_preds += (preds == labels).sum().item()
            total_preds += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = correct_preds / total_preds

    print(f"Época {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}%")

Utilizando o dispositivo: cuda
Época 1/3 | Train Loss: 0.5525 | Val Loss: 0.2727 | Val Acc: 91.42%
Época 2/3 | Train Loss: 0.1542 | Val Loss: 0.1748 | Val Acc: 94.83%
Época 3/3 | Train Loss: 0.0758 | Val Loss: 0.1853 | Val Acc: 94.55%


## Avaliação

In [ ]:
""" TODO: calcule métricas de classificação final no dataset de teste. Seria legal ver métricas por classe (livro)! """
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

# Mapeamento inverso para recuperar o nome do livro a partir do ID
id2label = {v: k for k, v in full_dataset.label2id.items()}
target_names = [id2label[i] for i in range(len(id2label))]

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Exibe as métricas completas de classificação (Precision, Recall, F1-Score por livro)
print("Relatório de Classificação no Dataset de Teste:\n")
print(classification_report(all_labels, all_preds, target_names=target_names))

Relatório de Classificação no Dataset de Teste:

                                 precision    recall  f1-score   support

                 a_mao_e_a_luva       0.99      0.82      0.90       379
                   dom_casmurro       0.92      0.89      0.91       277
                    esau_e_jaco       0.96      0.97      0.96       517
                         helena       0.95      0.88      0.91       218
                    iaia_garcia       0.88      0.98      0.93       319
              memorial_de_aires       0.98      1.00      0.99       330
memorias_postumas_de_bras_cubas       0.94      0.98      0.96       757
                  quincas_borba       0.92      0.95      0.93       455

                       accuracy                           0.94      3252
                      macro avg       0.94      0.93      0.94      3252
                   weighted avg       0.94      0.94      0.94      3252



## Exemplos


In [ ]:
""" TODO: selecione trechos aleatórios de teste e mostre alguns casos se o modelo está acertando """

import random

# PERGUNTAS DE DESENVOLVIMENTO:
# - Por que foi adicionado unsqueeze(0) nas amostras de teste?

# Resposta: Porque o modelo espera uma entrada em lotes (batch), ou seja, com formato [batch_size, sequence_length].
# Como estamos testando uma única amostra por vez, o unsqueeze(0) simula um batch de tamanho 1.

# Seleciona 5 índices aleatórios do dataset de teste para inspeção
test_indices = random.sample(range(len(test_data)), min(5, len(test_data)))

model.eval()
print("--- EXEMPLOS DE CLASSIFICAÇÃO EM TEMPO REAL ---\n")

with torch.no_grad():
    for idx in test_indices:
        sample = test_data[idx]
        input_ids = sample['input_ids'].unsqueeze(0).to(device)       # Adiciona dimensão de batch
        attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
        true_label_id = sample['label'].item()

        # Faz a predição
        logits = model(input_ids, attention_mask)
        pred_label_id = torch.argmax(logits, dim=-1).item()

        # Decodifica os tokens de volta para texto para ler o trecho
        decoded_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)

        real_book = id2label[true_label_id]
        predicted_book = id2label[pred_label_id]
        status = "ACERTOU" if true_label_id == pred_label_id else "ERROU"

        print(f"Trecho: \"{decoded_text[:120]}...\"")
        print(f"-> Livro Real: {real_book}")
        print(f"-> Predição do Modelo: {predicted_book} [{status}]\n" + "-"*50)

--- EXEMPLOS DE CLASSIFICAÇÃO EM TEMPO REAL ---

Trecho: "##ômodo, a agitação da véspera, o jantar tarde... Depois, pediu que lhe deixasse acabar de ler um artigo relativo a cert..."
-> Livro Real: quincas_borba
-> Predição do Modelo: quincas_borba [ACERTOU]
--------------------------------------------------
Trecho: "opostos, um de estima, outro de inveja, Pedro e Paulo concluíram ter recebido com ele um mérito especial. Quando, mais t..."
-> Livro Real: esau_e_jaco
-> Predição do Modelo: esau_e_jaco [ACERTOU]
--------------------------------------------------
Trecho: "##t desire them, then? PROMETHEUS. They were wrought by a divine hand. Vulcan forged them. Two emissaries from heaven ca..."
-> Livro Real: a_mao_e_a_luva
-> Predição do Modelo: a_mao_e_a_luva [ACERTOU]
--------------------------------------------------
Trecho: "outros na labutação, caminhando de tentativa em tentativa, entre ilusões e desilusões, para o mesmo termo comum. E é pou..."
-> Livro Real: memorias_postumas_de_

 # Célula para Selecionar Aleatoriamente trechos dos livros

In [ ]:
import os
import random

# 1. Diretório onde estão os livros
DATA_DIR = "livros_com_nomes"

# 2. Lista todos os arquivos .txt disponíveis na pasta
livros_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.txt')]

if livros_files:
    # Sorteia aleatoriamente um dos livros da pasta
    arquivo_escolhido = random.choice(livros_files)
    nome_livro = arquivo_escolhido.replace('.txt', '')
    caminho_arquivo = os.path.join(DATA_DIR, arquivo_escolhido)

    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        conteudo_livro = f.read()

    # Remove quebras de linha excessivas e múltiplos espaços para o trecho ficar bonito de ler
    conteudo_limpo = " ".join(conteudo_livro.split())

    # Sorteia uma posição segura no meio do texto para pegar o trecho
    tamanho_trecho = 500  # Tamanho em caracteres
    if len(conteudo_limpo) > tamanho_trecho:
        ponto_inicio = random.randint(0, len(conteudo_limpo) - tamanho_trecho)
        trecho_sorteado = conteudo_limpo[ponto_inicio : ponto_inicio + tamanho_trecho]
    else:
        trecho_sorteado = conteudo_limpo

    print(f"LIVRO SORTEADO: {nome_livro.upper()}\n")
    print(f"\"...{trecho_sorteado.strip()}...\"\n")
    print("-" * 60)
else:
    print("Nenhum arquivo .txt encontrado na pasta 'livros_com_nomes'. Verifique o diretório.")

LIVRO SORTEADO: QUINCAS_BORBA

"...s, Rubião, cansado de estar a olhar para o teto, enquanto o outro lhe aperfeiçoava os queixos, pedia para descansar. Descansando, apalpava o rosto e sentia pelo tacto a mudança. Os bigodes é que não estão muito compridos, observava. Falta arranjarlhe as guias; aqui trago os ferrinhos para encurválos bem sobre o lábio, e depois faremos as guias. Ah! eu prefiro compor dez trabalhos originais a uma só cópia. Volveram ainda dez minutos, antes que os bigodes e a pêra fossem bem retocados. Enfim, pron..."

------------------------------------------------------------


# Célula de Validação dos Trechos Inseridos Manualmente

In [ ]:
def prever_livro(texto_personalizado):
    model.eval()

    # 1. Tokeniza o texto recebido como parâmetro
    encoding = tokenizer(
        texto_personalizado,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 2. Faz a inferência com o modelo treinado
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(logits, dim=-1).item()
        confianca = probs[0][pred_id].item() * 100

    # 3. Recupera o nome do livro previsto
    id2label = {v: k for k, v in full_dataset.label2id.items()}
    livro_previsto = id2label[pred_id]

    print(f"Texto analisado: \"{texto_personalizado}\"")
    print(f"Livro Previsto pelo Modelo: ➔ {livro_previsto.upper()} (Confiança: {confianca:.2f}%)")

    # MOSTRAR OS TOKENS REAIS DENTRO DA FUNÇÃO:
    tokens_reais = tokenizer.convert_ids_to_tokens(input_ids[0])
    print(f"\nTokens lidos de fato pelo BERT ({len(tokens_reais)} tokens):\n{tokens_reais[:30]}...")

# --- TESTE ---
trecho_teste = "Havia já dois anos que nos não víamos, e eu via-a agora não qual era, mas qual fora, quais fôramos ambos, porque um Ezequias misterioso fizera recuar o sol até os dias juvenis."

prever_livro(trecho_teste)

Texto analisado: "Havia já dois anos que nos não víamos, e eu via-a agora não qual era, mas qual fora, quais fôramos ambos, porque um Ezequias misterioso fizera recuar o sol até os dias juvenis."
Livro Previsto pelo Modelo: ➔ MEMORIAS_POSTUMAS_DE_BRAS_CUBAS (Confiança: 99.91%)

Tokens lidos de fato pelo BERT (128 tokens):
['[CLS]', 'Havia', 'já', 'dois', 'anos', 'que', 'nos', 'não', 'ví', '##amos', ',', 'e', 'eu', 'via', '-', 'a', 'agora', 'não', 'qual', 'era', ',', 'mas', 'qual', 'fora', ',', 'quais', 'f', '##ô', '##ramos', 'ambos']...


# Probabilidades Calculadas pelo Modelo por Livro

In [ ]:
def inspecionar_probabilidades(texto_personalizado):
    model.eval()

    encoding = tokenizer(
        texto_personalizado,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=-1)[0]

    # Exibe a probabilidade atribuída a cada livro
    print("Probabilidades calculadas pelo modelo por livro:\n")
    id2label = {v: k for k, v in full_dataset.label2id.items()}

    for idx, prob in enumerate(probs):
        nome_livro = id2label[idx]
        percentual = prob.item() * 100
        print(f" - {nome_livro:<32}: {percentual:.2f}%")

inspecionar_probabilidades(trecho_teste)

Probabilidades calculadas pelo modelo por livro:

 - a_mao_e_a_luva                  : 0.01%
 - dom_casmurro                    : 0.04%
 - esau_e_jaco                     : 0.00%
 - helena                          : 0.02%
 - iaia_garcia                     : 0.01%
 - memorial_de_aires               : 0.00%
 - memorias_postumas_de_bras_cubas : 99.91%
 - quincas_borba                   : 0.01%


## Discussão

O classifcador desenvolvido nesse trabalho utiliza o BERTimbau e uma MLP no token [CLS]. Implementei o dataset com janelas deslizantes e o laço de treinamento manual do zero. Tive um problema inicial com um erro de pré-processamento que descartava acentos e maiúsculas, mas ajustei a classe para preservar a formatação nativa do tokenizador. Com essa correção, o modelo estabilizou e atingiu 94% de acurácia no conjunto de teste após 3 épocas.

## Disclaimer: Uso de IA

Declaro que fiz o uso do Gemini (3.5) como auxilio para o desenvolvimento dessa atividade.